# 03 — Group Comparisons (the statistics core)

Runs the three required hypothesis tests using `src/stats_utils.py` (the same functions `src/run_hypothesis_tests.py` uses to produce the saved results the Streamlit dashboard displays). For every test: H0/H1 are stated explicitly, the relevant assumption is checked (and the resulting test choice justified), and the result is interpreted in plain language — not just reported as a p-value.

**Note:** a fourth item, logistic regression for completion, is listed as an *optional stretch goal* in the roadmap PDF, but the build prompt's hard constraints explicitly rule out **any** model-fitting step ("No machine learning of any kind"). Per those constraints, which take priority, it is not implemented here.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from src.stats_utils import (
    test_phase_vs_completion,
    test_sponsor_vs_enrollment,
    test_duration_completed_vs_terminated,
)

df = pd.read_csv('../data/processed/analysis_table.csv',
                  parse_dates=['start_date', 'completion_date'])

## Test 1 — Chi-square test of independence: phase vs. completion status

**H0:** trial phase and overall completion status are statistically independent. **H1:** they are associated.

Both variables are categorical, so this is the standard test for association between two categorical variables. Assumption checked: expected cell counts should be ≥ 5 in every cell for the chi-square approximation to hold — checked explicitly below rather than assumed.

In [2]:
result1, contingency = test_phase_vs_completion(df)
display(contingency)
print()
for k, v in result1.to_dict().items():
    print(f'{k}: {v}')

overall_status,"Active, not recruiting",Completed,Recruiting,Terminated,Unknown status,Withdrawn
phase,,,,,,
Phase 1,261,1075,409,416,219,135
Phase 2,338,1577,397,368,233,148
Phase 3,194,1306,275,162,172,111
Phase 4,98,769,146,57,82,52



test_name: Chi-square test of independence: phase vs. overall_status
h0: Trial phase and overall completion status are statistically independent.
h1: Trial phase and overall completion status are associated.
statistic: 270.30960050822176
p_value: 7.989608592135053e-49
effect_size_name: Cramer's V
effect_size: 0.10005731700125531
n: 9000
interpretation: Phase and completion status are statistically dependent (chi2=270.3, p=7.99e-49); the association is weak in practical terms (Cramer's V=0.100), meaning phase shifts the completion-status mix somewhat but does not determine it.
assumption_note: All expected cell counts >= 5 (minimum observed: 59.7), so the chi-square approximation is appropriate.


## Test 2 — Sponsor type vs. enrollment size

**H0:** central enrollment is equal across Industry / NIH / Other sponsors. **H1:** at least one differs.

This is a continuous outcome (enrollment) compared across more than two categorical groups (sponsor type) — the standard case for a one-way ANOVA, *provided* its normality assumption holds. Since enrollment is right-skewed (Notebook 02), we check normality with Shapiro-Wilk per group first and fall back to the non-parametric Kruskal-Wallis test if it's violated — the cell below reports which test actually ran and why.

In [3]:
result2, sponsor_groups = test_sponsor_vs_enrollment(df)
display(sponsor_groups)
print()
for k, v in result2.to_dict().items():
    print(f'{k}: {v}')

,sponsor_type,count,mean,median,std
0,Industry,4900,177.073673,88.0,279.794090
1,NIH,647,159.939722,63.0,253.181993
2,Other,3453,115.700840,52.0,215.694476



test_name: Kruskal-Wallis H-test: sponsor_type vs. enrollment
h0: Mean/central enrollment is equal across Industry, NIH, and Other-sponsored trials.
h1: Enrollment differs across at least one sponsor type.
statistic: 381.27808206055533
p_value: 1.6088532984187987e-83
effect_size_name: epsilon-squared (rank-based)
effect_size: 0.04215606113821889
n: 9000
interpretation: Kruskal-Wallis H-test finds enrollment size differs across sponsor types (statistic=381.28, p=1.609e-83); the effect size is small (epsilon-squared (rank-based)=0.042). Median enrollment by sponsor type: Industry=88, NIH=63, Other=52.
assumption_note: Shapiro-Wilk rejected normality (p < 0.05) for: Industry, NIH, Other -- expected, since enrollment is right-skewed (see descriptive stats). ANOVA's normality assumption is not met, so the non-parametric Kruskal-Wallis test was used instead.


## Test 3 — Duration: Completed vs. Terminated/Withdrawn

**H0:** trial duration is equal between Completed trials and Terminated/Withdrawn trials. **H1:** duration differs.

A continuous outcome (`duration_days`) across exactly two groups is the standard case for an independent t-test, again conditional on (approximate) normality; otherwise Mann-Whitney U is the non-parametric alternative. Rows with no `duration_days` (still-ongoing trials) are excluded here only, exactly as documented in Notebook 01 and `docs/methodology.md`.

In [4]:
result3, duration_groups = test_duration_completed_vs_terminated(df)
display(duration_groups)
print()
for k, v in result3.to_dict().items():
    print(f'{k}: {v}')

,group,count,mean,median,std
0,Completed,4727,817.855088,796.0,314.477080
1,Terminated/Withdrawn,1449,350.766736,323.0,155.141308



test_name: Mann-Whitney U test: duration_days, Completed vs. Terminated/Withdrawn
h0: Trial duration is equal between Completed and Terminated/Withdrawn trials.
h1: Trial duration differs between Completed and Terminated/Withdrawn trials.
statistic: 6265539.0
p_value: 0.0
effect_size_name: rank-biserial correlation
effect_size: -0.8295085586041335
n: 6176
interpretation: Mann-Whitney U test finds duration differs significantly between Completed and Terminated/Withdrawn trials (statistic=6265539.00, p=0); Completed trials run longer on average (818 vs 351 median-adjacent days), rank-biserial correlation=-0.830.
assumption_note: Shapiro-Wilk rejected normality for at least one group (Completed p=4.15e-22, Terminated/Withdrawn p=4.13e-23), so the non-parametric Mann-Whitney U test was used instead of a t-test.


## Save results table

Writes the same `results/hypothesis_tests.csv` / `.json` that `src/run_hypothesis_tests.py` produces, so the Streamlit dashboard can display these results without recomputing anything live.

In [5]:
results_df = pd.DataFrame([result1.to_dict(), result2.to_dict(), result3.to_dict()])
os.makedirs('../results', exist_ok=True)
results_df.to_csv('../results/hypothesis_tests.csv', index=False)
results_df[['test_name', 'statistic', 'p_value', 'effect_size_name', 'effect_size']]

,test_name,statistic,p_value,effect_size_name,effect_size
0,Chi-square test of independence: phase vs. ove...,2.703096e+02,7.989609e-49,Cramer's V,0.100057
1,Kruskal-Wallis H-test: sponsor_type vs. enroll...,3.812781e+02,1.608853e-83,epsilon-squared (rank-based),0.042156
2,"Mann-Whitney U test: duration_days, Completed ...",6.265539e+06,0.000000e+00,rank-biserial correlation,-0.829509
